In [50]:
import json
import urllib.request
import urllib.error
import random
import time

TEAM_ID = "TEAM_62"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"
API_KEY = "oc_EA1hJavBgDDikKyWX1kyu6PIxnl57glI"

def api(payload):
    headers = {
        "Content-Type": "application/json",
        "X-API-Key": API_KEY,
        "team_id": TEAM_ID
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data, headers=headers, method="POST")
    
    try:
        with urllib.request.urlopen(req) as response:
            response_body = response.read().decode("utf-8")
            return json.loads(response_body)
    except urllib.error.HTTPError as e:
        error_body = e.read().decode("utf-8")
        print(f"HTTP Error {e.code}: {e.reason}")
        print(f"Server response body: {error_body}")
        raise e

def oracle_query(params):
    payload = {
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    }
    
    time.sleep(1.0) # Rate-limit protection
    result = api(payload)
    return float(result["loss"])

print("API connection loaded successfully!")

API connection loaded successfully!


In [54]:
SPACE = {
    "tol": [0.001, 0.0001],
    "alpha": [
        0.0001, 0.00020691380811147902, 0.00042813323987193956, 0.0008858667904100822,
        0.0018329807108324356, 0.00379269019073225, 0.007847599703514606, 0.01623776739188721,
        0.03359818286283781, 0.06951927961775606, 0.14384498882876628, 0.29763514416313164,
        0.615848211066026, 1.2742749857031321, 2.6366508987303554, 5.455594781168514,
        11.288378916846883, 23.357214690901213, 48.32930238571752, 100
    ],
    "l1_ratio": [
        0.05, 0.17857142857142855, 0.3071428571428571, 0.43571428571428567,
        0.5642857142857143, 0.6928571428571428, 0.8214285714285714, 0.95
    ],
    "positive": [True, False],
    "selection": ["cyclic", "random"],
    "fit_intercept": [True, False]
}

MAX_GLOBAL_CALLS = 50
MAX_LOCAL_CALLS = 10

best_loss = float('inf')
best_params = None

print("Official search space initialized!")

Official search space initialized!


In [ ]:
print("Starting Phase 1: Global Exploration...")
print("-" * 40)

for i in range(MAX_GLOBAL_CALLS):
    current_params = {
        "tol": random.choice(SPACE["tol"]),
        "alpha": random.choice(SPACE["alpha"]),
        "l1_ratio": random.choice(SPACE["l1_ratio"]),
        "positive": random.choice(SPACE["positive"]),
        "selection": random.choice(SPACE["selection"]),
        "fit_intercept": random.choice(SPACE["fit_intercept"])
    }
    
    loss = oracle_query(current_params)
    
    if loss < best_loss:
        best_loss = loss
        best_params = current_params.copy()
        print(f"[Iteration {i+1}] New Best Loss Found! -> {loss:.5f}")

print("-" * 40)
print(f"Phase 1 complete! Best loss so far: {best_loss:.5f}")

Starting Phase 1: Global Exploration...
----------------------------------------
[Iteration 1] New Best Loss Found! -> 198.99427
[Iteration 2] New Best Loss Found! -> 15.55869
[Iteration 3] New Best Loss Found! -> 8.89058
